# Stage 00b — Descriptions CSV + EasyOCR Figure Matching

**What this notebook does:**

1. Exports all *Brief Description of the Drawings* texts from the PatSeer Excel
   into a single `data/descriptions.csv` (replaces the old per-patent `.txt` files).

2. Assigns a figure label to every downloaded image using **EasyOCR spatial
   matching** — reads figure labels directly from the drawing sheets.

The matching logic is:
1. Run EasyOCR on the full image to detect figure labels (e.g. `FIG. 3A`)
2. Split the image at horizontal/vertical whitespace bands (≥15 px, >95% white)
3. Assign each crop the nearest detected label whose centre falls inside or ≤80 px below it
4. If EasyOCR finds nothing: positional fallback — match by document order if counts agree,
   else flag everything `_Fu` (needs human review via Stage 01)
5. FAT files are always split and always `_Fu`

**Output naming:**

| Label source | Output filename | Meaning |
|---|---|---|
| EasyOCR / positional | `US…_F003A.png` | matched to FIG. 3A |
| Unmatched / FAT / count mismatch | `US…_Fu001.png` | needs review |

Output is written to `matched/<patent_id>/` — raw files are never modified.

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b  ←  YOU ARE HERE  (descriptions CSV + EasyOCR matching → _F / _Fu labels)
 ↓
01   (optional human review for _Fu files)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|------|
| 1 | Load config, Excel index, imports |
| 2 | Save `data/descriptions.csv` |
| 3 | Initialise EasyOCR reader (once per session) |
| 4 | Run pipeline (respects `scan_limit` from config) |
| 5 | Review crops flagged for human inspection |


In [2]:
import sys, importlib.util, re
import pandas as pd
from pathlib import Path

# Walk upward from cwd until we find the repo root (has both src/ and config.yaml).
# This handles VS Code running notebooks with cwd = workspace root.
_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")
src_dir = repo_root / "src"
print(f"repo_root : {repo_root}")
print(f"src_dir   : {src_dir}")

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

for p in [str(repo_root), str(src_dir)]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_dir))

for _mod in ["figure_matcher", "extractor"]:
    sys.modules.pop(_mod, None)

from figure_matcher import build_easyocr_reader, process_all_patents
from extractor import load_patseer_excel, save_descriptions_csv

cfg         = load_config()
excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
df          = pd.DataFrame(excel_index.values())
raw_dir     = cfg["paths"]["raw_images"]
print(f"Loaded {len(df)} patents from Excel.")


repo_root : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools
src_dir   : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [3]:
# ─── Save brief-description-of-drawings CSV ─────────────────────────────────
# Reads the 'description_of_drawings' column already loaded from the Excel
# and writes one CSV: data/descriptions.csv  (patent_id, description_of_drawings)
# This replaces the individual text/{patent_id}.txt files from the old pipeline.

data_dir  = cfg["paths"]["data"]
csv_out   = save_descriptions_csv(excel_index, data_dir / "descriptions.csv")

n_total = len(excel_index)
n_filled = sum(1 for m in excel_index.values() if m.get("description_of_drawings"))
print(f"Saved {n_total} rows → {csv_out}")
print(f"  {n_filled} patents have a description  |  {n_total - n_filled} are empty")


Saved 1639 rows → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/descriptions.csv
  1282 patents have a description  |  357 are empty


In [4]:
reader = build_easyocr_reader(gpu=False)
print("EasyOCR ready")


Using CPU. Note: This module is much faster with a GPU.


EasyOCR ready


In [5]:
# ─── Single-patent test ───────────────────────────────────────────────────────
# Change TEST_PATENT to any Excel patent ID; leave as None to use the first one.
TEST_PATENT = None

from figure_matcher import process_patent, _patent_core, _build_folder_map

raw_dir     = cfg["paths"]["raw_images"]
matched_dir = cfg["paths"]["matched"]
folder_map  = _build_folder_map(raw_dir)

excel_id = TEST_PATENT or df["patent_id"].iloc[0]
folder   = folder_map.get(_patent_core(excel_id))

if folder is None:
    print(f"No raw folder found for {excel_id}")
else:
    actual_id = folder.name
    desc      = df.loc[df["patent_id"] == excel_id, "description_of_drawings"].iloc[0]
    desc      = str(desc) if desc == desc else ""   # NaN guard

    print(f"Patent       : {excel_id}  (folder: {actual_id})")
    print(f"Raw input    : {raw_dir / actual_id}")
    print(f"Matched out  : {matched_dir / actual_id}")
    print()

    summary = process_patent(actual_id, raw_dir, matched_dir, desc, cfg, reader)

    print(f"total_crops  : {summary['total_crops']}")
    print(f"labeled      : {summary['labeled']}")
    print(f"unlabeled    : {summary['unlabeled']}")
    print(f"needs_review : {summary['needs_review']}")
    print()
    for f in summary["files"]:
        tag = "✓" if f["label"] else "⚠"
        print(f"  {tag}  {f['original']:<45}  →  {f['output']}")


Patent       : US2022267016A1  (folder: US20220267016A1)
Raw input    : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw/US20220267016A1
Matched out  : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/matched/US20220267016A1

total_crops  : 120
labeled      : 10
unlabeled    : 110
needs_review : True

  ⚠  US20220267016A1_D00000.png                     →  US20220267016A1_Fu001.png
  ⚠  US20220267016A1_D00000.png                     →  US20220267016A1_Fu002.png
  ⚠  US20220267016A1_D00000.png                     →  US20220267016A1_Fu003.png
  ⚠  US20220267016A1_D00000.png                     →  US20220267016A1_Fu004.png
  ⚠  US20220267016A1_D00000.png                     →  US20220267016A1_Fu005.png
  ⚠  US20220267016A1_D00000.png                     →  US20220267016A1_Fu006.png
  ⚠  US20220267016A1_D00000.png                     →  US20220267016A1_Fu007.png
  ⚠  US20220267016A1_D00001.png         

In [ ]:
scan_limit = cfg["extractor"].get("scan_limit", None)  # set to e.g. 5 in config for dev
test_df    = df.head(scan_limit) if scan_limit else df

results_df = process_all_patents(test_df, cfg, reader)
print(results_df.groupby("source_type")[["labeled","unlabeled"]].sum())


In [ ]:
flagged = results_df[results_df["needs_review"] == True]
print(f"{len(flagged)} crops flagged for human review")
display(flagged[["patent_id","original","output","label"]].head(20))
